# 🔬 scRNA-seq 기초 분석 실습
## Part 1: Quality Control & Preprocessing

**건국대학교 이형우 교수님 연구실 온라인 세미나**  
Data: GSE210543 (Human Retina scRNA-seq — Developmental vs Adult)

---

### 학습 목표
1. 10X Genomics 데이터를 Seurat으로 불러오기
2. QC 지표의 생물학적 의미 이해하기
3. QC 기준을 시각화하고 필터링 적용하기
4. 정규화 및 고변이 유전자 선택하기

### 분석 샘플
| 그룹 | 샘플 | 세포 수 |
|------|------|--------|
| Young (발달기) | 16PCW | 8,601 |
| Young (발달기) | 20PCW | 5,787 |
| Old (성체) | Adult_2 | 6,516 |
| Old (성체) | Adult_3 | 3,694 |

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_04.png" width="850"/>

*Fig. 0 — scRNA-seq 분석의 복잡성: 생물학적·기술적 변이 요인 (Hicks et al., BioRxiv 2015)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_05.png" width="850"/>

*Fig. 1 — scRNA-seq 전체 분석 워크플로우 (Luecken & Theis, Mol Syst Biol 2019)*

---
## Step 0. 환경 설정

> **주의:** 패키지 설치는 처음 한 번만 실행하면 됩니다. Colab 세션이 끊기면 재실행이 필요합니다.

In [ ]:
# 패키지 설치 (처음 실행 시에만 — 약 5~10분 소요)
if (!requireNamespace("Seurat", quietly = TRUE)) {
  install.packages("Seurat")
}
if (!requireNamespace("dplyr", quietly = TRUE)) {
  install.packages("dplyr")
}
if (!requireNamespace("ggplot2", quietly = TRUE)) {
  install.packages("ggplot2")
}
if (!requireNamespace("patchwork", quietly = TRUE)) {
  install.packages("patchwork")
}

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(patchwork)

set.seed(42)
cat("Seurat version:", as.character(packageVersion("Seurat")), "\n")

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_06.png" width="850"/>

*Fig. 2 — Seurat 기반 scRNA-seq 분석 워크플로우*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_07.png" width="850"/>

*Fig. 3 — Seurat vs Scanpy 비교 및 Tip 1: 사용자 환경에 맞게 선택하라*

---
## Step 1. Google Drive 연결 및 데이터 불러오기

### 데이터 구조 (10X Genomics filtered_feature_bc_matrix)

```
sample_folder/
├── barcodes.tsv.gz   ← 세포 바코드 목록 (행 = 세포)
├── features.tsv.gz   ← 유전자 목록 (열 = 유전자)
└── matrix.mtx.gz     ← 발현량 행렬 (sparse matrix)
```

> **10X Genomics 작동 원리**: 각 세포를 액적(droplet)에 가두고 고유한 바코드를 부여합니다.  
> mRNA는 역전사 후 시퀀싱되며, **UMI(Unique Molecular Identifier)**로 PCR 중복을 제거합니다.

In [ ]:
# Google Drive 마운트
# Colab 좌측 파일 아이콘 → Drive 마운트 버튼 클릭
# 또는 아래 실행 후 인증 링크 클릭

# R에서 Drive 마운트 명령 실행
system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

# 데이터 경로 설정 (공유된 Google Drive 경로로 수정 필요)
DATA_DIR <- "/content/drive/MyDrive/KU_seminar/GSE210543/filtered_feature_bc_matrix"
cat("Data directory:", DATA_DIR, "\n")

In [ ]:
# 4개 샘플 불러오기
samples <- list(
  Young_16PCW  = file.path(DATA_DIR, "16PCW"),
  Young_20PCW  = file.path(DATA_DIR, "20PCW"),
  Old_Adult2   = file.path(DATA_DIR, "Adult_2"),
  Old_Adult3   = file.path(DATA_DIR, "Adult_3")
)

# Seurat 오브젝트 생성
seurat_list <- lapply(names(samples), function(name) {
  cat("Loading:", name, "...\n")
  counts <- Read10X(data.dir = samples[[name]])
  obj <- CreateSeuratObject(
    counts  = counts,
    project = name,
    min.cells = 3,    # 최소 3개 세포에서 발현된 유전자만 포함
    min.features = 200  # 최소 200개 유전자가 발현된 세포만 포함
  )
  obj$sample <- name
  obj$group  <- ifelse(grepl("Young", name), "Young", "Old")
  return(obj)
})
names(seurat_list) <- names(samples)

# 각 샘플 기본 정보 확인
for (name in names(seurat_list)) {
  obj <- seurat_list[[name]]
  cat(sprintf("%s: %d cells, %d genes\n", name, ncol(obj), nrow(obj)))
}

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_08.png" width="850"/>

*Fig. 4 — Count Matrix 생성 원리: barcodes / features / matrix (Macosko et al., Cell 2015)*

---
## Step 2. QC (Quality Control)

### 🧬 QC 지표의 생물학적 의미

scRNA-seq 데이터에는 **저품질 세포(low-quality cells)**가 섞여 있습니다.  
이를 제거하지 않으면 분석 결과가 왜곡됩니다.

---

#### 1️⃣ nFeature_RNA — 세포당 검출된 유전자 수

```
정상 세포:  200 ~ 6,000개 유전자
빈 액적:    < 200개    → 세포가 없는 빈 droplet
이중 세포:  매우 높음   → 두 세포가 하나로 잡힌 doublet
```

#### 2️⃣ nCount_RNA (nUMI) — 세포당 전체 UMI 수

```
정상 세포:  500 ~ 50,000 UMI
저품질:    < 500       → RNA 포획 실패 또는 세포 사멸
이중 세포:  매우 높음   → doublet 의심
```

> **UMI(Unique Molecular Identifier)**: PCR 증폭 편향을 보정하는 고유 분자 바코드.  
> 동일한 UMI를 가진 read는 같은 분자에서 온 것으로 간주하여 1개로 계수합니다.

#### 3️⃣ percent.mt — 미토콘드리아 유전자 비율

```
정상 세포:  < 10%
손상/사멸:  > 10%  → 세포막 손상 시 세포질 RNA가 빠져나가지만
                      미토콘드리아는 막으로 둘러싸여 있어 RNA가 남음
```

> **왜 미토콘드리아인가?**  
> 세포가 죽거나 스트레스를 받으면 세포질의 RNA가 용해되어 빠져나갑니다.  
> 하지만 미토콘드리아는 자체 막을 가지고 있어 RNA가 남아있습니다.  
> 따라서 mt% 비율이 높을수록 손상된 세포를 의미합니다.

---

### 📊 이번 세미나 QC 기준 (Default Threshold)

| 지표 | 기준 | 의미 |
|------|------|------|
| nFeature_RNA | **> 200** | 최소 200개 유전자 발현 세포만 유지 |
| nCount_RNA | **> 500** | 최소 500 UMI 이상 세포만 유지 |
| percent.mt | **< 10%** | 미토콘드리아 비율 10% 미만 |

> **Tip**: 이 기준은 고정된 것이 아닙니다. 조직 유형, 실험 조건에 따라 조정이 필요합니다.  
> 시각화 후 데이터 분포를 보고 결정하는 것이 좋습니다.

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_09.png" width="850"/>

*Fig. 5 — QC 지표 분포 확인 방법 (NCells, nUMI, nGene, mitoRatio)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_10.png" width="850"/>

*Fig. 6 — QC VlnPlot 예시: nFeature_RNA / nCount_RNA / percent.mt*

In [ ]:
# 미토콘드리아 유전자 비율 계산
# 인간 미토콘드리아 유전자는 'MT-' 로 시작
seurat_list <- lapply(seurat_list, function(obj) {
  obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
  return(obj)
})

# QC 지표 확인 (첫 번째 샘플 예시)
head(seurat_list[[1]]@meta.data[, c("nFeature_RNA", "nCount_RNA", "percent.mt")])

In [ ]:
# QC 지표 시각화 — Violin Plot
# 각 샘플별로 분포를 확인합니다

plot_list <- lapply(names(seurat_list), function(name) {
  VlnPlot(
    seurat_list[[name]],
    features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
    ncol = 3,
    pt.size = 0
  ) + plot_annotation(title = name)
})

# 샘플별 출력
for (p in plot_list) print(p)

In [ ]:
# nFeature vs nCount 산점도 — doublet 탐지
# 정상 세포는 선형 관계를 보임
# 이상치(doublet)는 오른쪽 상단에 위치

scatter_list <- lapply(names(seurat_list), function(name) {
  p1 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "percent.mt"
  ) + ggtitle(paste(name, "- Count vs MT%"))

  p2 <- FeatureScatter(
    seurat_list[[name]],
    feature1 = "nCount_RNA",
    feature2 = "nFeature_RNA"
  ) + ggtitle(paste(name, "- Count vs Feature"))

  p1 + p2
})

for (p in scatter_list) print(p)

### 🔍 QC 분포 요약 통계

시각화 후, 각 샘플의 분포를 수치로도 확인합니다.  
특히 **중앙값(median)**과 **이상치 범위**를 주목하세요.

In [ ]:
# 샘플별 QC 요약 통계
qc_summary <- do.call(rbind, lapply(names(seurat_list), function(name) {
  meta <- seurat_list[[name]]@meta.data
  data.frame(
    Sample         = name,
    Group          = unique(meta$group),
    Cells          = nrow(meta),
    nFeature_median = median(meta$nFeature_RNA),
    nFeature_max    = max(meta$nFeature_RNA),
    nCount_median   = median(meta$nCount_RNA),
    mt_median       = round(median(meta$percent.mt), 2),
    mt_max          = round(max(meta$percent.mt), 2)
  )
}))

print(qc_summary)

---
## Step 3. QC 필터링 적용

시각화로 분포를 확인한 후, 아래 기준으로 저품질 세포를 제거합니다.

| 지표 | 기준 |
|------|------|
| nFeature_RNA | **> 200** |
| nCount_RNA | **> 500** |
| percent.mt | **< 10%** |

> **💡 Discussion**: 위 VlnPlot을 보고 이 기준이 적절한지 토론해봅시다.  
> - 이 샘플에서 특이한 분포가 보이는 샘플이 있나요?
> - 더 엄격한 기준이 필요한 경우는 언제일까요?

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_11.png" width="850"/>

*Fig. 7 — QC 필터링 실습 코드 및 Tip 2: Filter 조건에서의 정답은 없다!*

In [ ]:
# QC 기준값 설정 (여기서 조정 가능)
QC_MIN_FEATURE <- 200   # 최소 유전자 수
QC_MIN_COUNT   <- 500   # 최소 UMI 수
QC_MAX_MT      <- 10    # 최대 미토콘드리아 비율 (%)

# 필터링 적용
seurat_list_filtered <- lapply(names(seurat_list), function(name) {
  obj <- seurat_list[[name]]
  before <- ncol(obj)

  obj <- subset(
    obj,
    subset = nFeature_RNA > QC_MIN_FEATURE &
             nCount_RNA   > QC_MIN_COUNT   &
             percent.mt   < QC_MAX_MT
  )

  after <- ncol(obj)
  removed <- before - after
  cat(sprintf("%s: %d → %d cells (removed %d, %.1f%%)\n",
              name, before, after, removed, removed/before*100))
  return(obj)
})
names(seurat_list_filtered) <- names(seurat_list)

In [ ]:
# 필터링 후 QC 분포 재확인
plot_list_after <- lapply(names(seurat_list_filtered), function(name) {
  VlnPlot(
    seurat_list_filtered[[name]],
    features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
    ncol = 3,
    pt.size = 0
  ) + plot_annotation(title = paste(name, "[After QC]"))
})

for (p in plot_list_after) print(p)

---
## Step 4. 정규화 (Normalization)

### 🧬 왜 정규화가 필요한가?

각 세포마다 포획된 RNA의 총량이 다릅니다.  
이를 보정하지 않으면 **RNA 포획 효율의 차이**가 마치 **생물학적 차이**처럼 보입니다.

#### LogNormalize 방법
```
normalized = log( (count / total_count_per_cell) × scale_factor + 1 )
```
- `scale_factor`: 기본값 10,000 (CPM과 유사)
- `log1p`: 0인 값을 처리하고 분포를 정규화

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_12.png" width="850"/>

*Fig. 8 — LogNormalization vs SCTransform 방법 비교 및 PC 선택 기준 (Tip 3)*

<img src="https://raw.githubusercontent.com/JeonghanSeo/KU-scRNAseq-seminar/main/figures/slide_13.png" width="850"/>

*Fig. 9 — 정규화 개념: SCTransform vs LogNormalization (M. Loven, RNA-seq statistical analysis)*

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  NormalizeData(
    obj,
    normalization.method = "LogNormalize",
    scale.factor = 10000
  )
})

cat("Normalization complete!\n")

---
## Step 5. 고변이 유전자 선택 (Highly Variable Genes, HVG)

### 🧬 왜 HVG를 선택하는가?

인간 게놈의 약 20,000개 유전자 중 대부분은 **모든 세포에서 비슷하게 발현**됩니다.  
세포 유형을 구분하는 데 유용한 유전자는 **세포마다 발현량이 크게 다른 유전자**입니다.

- 너무 적게 발현: 노이즈가 많음
- 모든 세포에서 균일하게 발현: 정보 없음
- **세포마다 다르게 발현**: 세포 유형 구분에 유용 ✓

**기본값: 상위 2,000개 HVG 선택**

In [ ]:
seurat_list_filtered <- lapply(seurat_list_filtered, function(obj) {
  FindVariableFeatures(
    obj,
    selection.method = "vst",
    nfeatures = 2000
  )
})

# Top 10 HVG 확인 (첫 번째 샘플)
top10 <- head(VariableFeatures(seurat_list_filtered[[1]]), 10)
cat("Top 10 HVGs (16PCW):\n")
print(top10)

In [ ]:
# HVG 시각화
plot_hvg_list <- lapply(names(seurat_list_filtered), function(name) {
  obj  <- seurat_list_filtered[[name]]
  top10 <- head(VariableFeatures(obj), 10)
  p <- VariableFeaturePlot(obj)
  LabelPoints(plot = p, points = top10, repel = TRUE) +
    ggtitle(name)
})

for (p in plot_hvg_list) print(p)

---
## Step 6. 오브젝트 병합 및 저장

QC와 정규화가 완료된 4개 샘플을 하나의 Seurat 오브젝트로 병합합니다.  
다음 파트(통합 분석)에서 사용합니다.

In [ ]:
# 4개 샘플 병합
seurat_merged <- merge(
  seurat_list_filtered[[1]],
  y    = seurat_list_filtered[2:4],
  add.cell.ids = names(seurat_list_filtered)
)

# 결과 확인
cat("Merged object:\n")
print(seurat_merged)
cat("\nSample composition:\n")
print(table(seurat_merged$sample))
cat("\nGroup composition:\n")
print(table(seurat_merged$group))

In [ ]:
# Google Drive에 저장 (다음 파트에서 로드 가능)
SAVE_DIR <- "/content/drive/MyDrive/KU_seminar/results"
dir.create(SAVE_DIR, showWarnings = FALSE, recursive = TRUE)

saveRDS(seurat_merged, file = file.path(SAVE_DIR, "01_seurat_merged_after_QC.rds"))
cat("Saved:", file.path(SAVE_DIR, "01_seurat_merged_after_QC.rds"), "\n")

---
## ✅ Part 1 완료!

### 정리

| 단계 | 내용 |
|------|------|
| 데이터 로딩 | Read10X → CreateSeuratObject |
| QC 계산 | nFeature, nCount, percent.mt |
| QC 필터링 | nFeature > 200, nCount > 500, mt < 10% |
| 정규화 | LogNormalize (scale 10,000) |
| HVG 선택 | 상위 2,000개 유전자 |
| 저장 | .rds 파일로 Google Drive 저장 |

### 다음 파트
**Part 2: Integration, Clustering & UMAP**  
→ Young vs Old 샘플을 통합하고 세포 클러스터를 분류합니다.

---
*GSE210543 | Human Retina scRNA-seq | KU Online Seminar*